# 06 — Machine Learning Demand Prediction
Train all models, compare metrics, SHAP explainability, and live prediction.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd

metrics = pd.read_csv('../outputs/metrics/model_metrics.csv')
test_m = metrics[metrics['split'] == 'test'].sort_values('R2', ascending=False)
print('Model Comparison (Test Set):')
test_m[['model_name','MAE','RMSE','MAPE','R2']]

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
img = Image.open('../reports/figures/model_comparison.png')
plt.figure(figsize=(14, 10))
plt.imshow(img)
plt.axis('off')
plt.title('Model Performance Comparison')
plt.show()

In [ ]:
shap_imp = pd.read_csv('../outputs/metrics/shap_feature_importance.csv')
print('Top 10 demand drivers (SHAP):')
print(shap_imp.head(10).to_string(index=False))

In [ ]:
img2 = Image.open('../reports/figures/shap_global_importance.png')
plt.figure(figsize=(10, 7))
plt.imshow(img2)
plt.axis('off')
plt.title('SHAP Global Feature Importance')
plt.show()

In [ ]:
img3 = Image.open('../reports/figures/shap_summary_plot.png')
plt.figure(figsize=(11, 8))
plt.imshow(img3)
plt.axis('off')
plt.title('SHAP Summary Plot')
plt.show()

In [ ]:
from src.ml.predict import predict_single

result = predict_single(
    route='Kurnool-Hyderabad',
    date='2024-12-15',
    bus_type='Volvo Ac',
    distance_km=326.0,
    capacity=49,
    is_holiday=0,
    bus_capacity=50,
)
print('Prediction Result:')
for k, v in result.items():
    print(f'  {k:25s}: {v}')

In [ ]:
# Actual vs Predicted (test set)
pred_df = pd.read_csv('../outputs/predictions/demand_predictions.csv')
test_pred = pred_df[pred_df['split'] == 'test']

plt.figure(figsize=(10, 6))
plt.scatter(test_pred['actual_demand'], test_pred['predicted_demand'],
            alpha=0.6, color='steelblue', s=35)
mn, mx = test_pred['actual_demand'].min(), test_pred['actual_demand'].max()
plt.plot([mn, mx], [mn, mx], 'r--', label='Perfect Fit')
plt.xlabel('Actual Passengers')
plt.ylabel('Predicted Passengers')
plt.title('Actual vs Predicted Demand (Test Set)')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/actual_vs_predicted.png', dpi=150)
plt.show()

## SHAP Findings

Top demand drivers identified by SHAP:
1. **capacity** — Bus type capacity is the strongest predictor
2. **rolling_mean_14** — 14-day historical average (strongest temporal predictor)
3. **bus_type_encoded** — Service class significantly affects demand
4. **week_of_year** — Seasonal patterns within the year
5. **fare_per_passenger** — Price elasticity effect

> Note: With only 1,000 records and 15 routes (~67 records/route),
> tree models require hyperparameter tuning to generalise beyond training data.